# Solution A — Evaluation on Dev Set

Loads the saved model bundle produced by `solution_A_train.ipynb` and performs detailed evaluation on the dev set: metrics comparison, confusion matrices, per-class analysis, and structured error analysis.

In [1]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'sklearn': 'scikit-learn',
    'joblib': 'joblib',
}

missing_packages = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
else:
    print('All required packages are already available.')


All required packages are already available.


## 1. Imports and path resolution

The notebook should work whether it is run from the repository root or from inside `solution-a/`.

The path helper below searches upward until it finds the coursework root that contains both:

- `training_data/NLI/train.csv`
- `nlu_bundle-feature-unified-local-scorer/`

This avoids the broken relative-path problem that the earlier notebook had.


In [2]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MaxAbsScaler, StandardScaler
from sklearn.svm import LinearSVC

SEED = 42


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (
            (candidate / 'training_data' / 'NLI' / 'train.csv').exists()
            and (candidate / 'nlu_bundle-feature-unified-local-scorer').exists()
        ):
            return candidate
    raise FileNotFoundError(
        'Could not find the coursework root. Expected to find training_data/NLI/train.csv '
        'and nlu_bundle-feature-unified-local-scorer/ in the same project tree.'
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / 'solution-a'
TRAIN_PATH = PROJECT_ROOT / 'training_data' / 'NLI' / 'train.csv'
DEV_PATH = PROJECT_ROOT / 'training_data' / 'NLI' / 'dev.csv'
TRIAL_PATH = PROJECT_ROOT / 'trial_data' / 'NLI_trial.csv'
LOCAL_SCORER_ROOT = PROJECT_ROOT / 'nlu_bundle-feature-unified-local-scorer'
OFFICIAL_BASELINE_PATH = LOCAL_SCORER_ROOT / 'baseline' / '25_DEV_NLI.csv'
ARTEFACT_DIR = NOTEBOOK_DIR / 'artifacts_solution_a'
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('TRAIN_PATH   =', TRAIN_PATH)
print('DEV_PATH     =', DEV_PATH)
print('TRIAL_PATH   =', TRIAL_PATH)
print('ARTEFACT_DIR =', ARTEFACT_DIR)


PROJECT_ROOT = /Users/jiho/Documents/YR3/34812NLU/NLU_CW
TRAIN_PATH   = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/training_data/NLI/train.csv
DEV_PATH     = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/training_data/NLI/dev.csv
TRIAL_PATH   = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/trial_data/NLI_trial.csv
ARTEFACT_DIR = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/solution-a/artifacts_solution_a


## 2. Data loading

The helper below accepts both training / dev CSVs and trial / test CSVs.

Rules used here:

- `premise` and `hypothesis` are required
- `label` is required only when `require_label=True`
- the helper strips a UTF-8 BOM from the first column name, which is useful for the supplied trial file


In [3]:
def read_pair_dataframe(csv_path: Path, require_label: bool) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [str(col).lstrip('﻿').strip() for col in df.columns]

    required_columns = {'premise', 'hypothesis'}
    missing_required = required_columns - set(df.columns)
    if missing_required:
        raise ValueError(f'{csv_path} is missing required columns: {sorted(missing_required)}')

    if require_label and 'label' not in df.columns:
        raise ValueError(f'{csv_path} must contain a label column for this section of the notebook.')

    if 'label' in df.columns:
        df['label'] = df['label'].astype(int)

    return df


train_df = read_pair_dataframe(TRAIN_PATH, require_label=True)
dev_df = read_pair_dataframe(DEV_PATH, require_label=True)
trial_df = read_pair_dataframe(TRIAL_PATH, require_label=True) if TRIAL_PATH.exists() else None

print('Train rows:', len(train_df))
print('Dev rows:  ', len(dev_df))
print('Trial rows:', len(trial_df) if trial_df is not None else 'trial file not found')
print()
print('Train label distribution:')
print(train_df['label'].value_counts().sort_index())
print()
train_df.head(3)


Train rows: 24432
Dev rows:   6736
Trial rows: 50

Train label distribution:
label
0    11784
1    12648
Name: count, dtype: int64



,premise,hypothesis,label
0,yeah i don't know cut California in half or so...,Yeah. I'm not sure how to make that fit. Maybe...,1
1,actual names will not be used,"For the sake of privacy, actual names are not ...",1
2,The film was directed by Randall Wallace.,The film was directed by Randall Wallace and s...,1


## 3. Evaluation helpers and official baseline table

The local scorer ships with the following metric names for NLI dev evaluation:

- `accuracy_score`
- `macro_precision`
- `macro_recall`
- `macro_f1`
- `weighted_macro_precision`
- `weighted_macro_recall`
- `weighted_mmacro_f1`
- `matthews_corrcoef`

In this notebook, **model selection is driven by `macro_f1`** rather than plain accuracy.
`macro_f1` gives equal weight to both labels, so it is a better fit than plain accuracy when we want balanced performance.

For Category A comparison, we also load the bundled `SVM` predictions from `25_DEV_NLI.csv`.


In [4]:
METRIC_ORDER = [
    'accuracy_score',
    'macro_precision',
    'macro_recall',
    'macro_f1',
    'weighted_macro_precision',
    'weighted_macro_recall',
    'weighted_mmacro_f1',
    'matthews_corrcoef',
]


def metric_summary(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        'accuracy_score': accuracy_score(y_true, y_pred),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_macro_precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'weighted_macro_recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'weighted_mmacro_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'matthews_corrcoef': matthews_corrcoef(y_true, y_pred),
    }


def metrics_table(predictions: dict[str, np.ndarray], y_true: np.ndarray) -> pd.DataFrame:
    rows = []
    for model_name, y_pred in predictions.items():
        row = {'model': model_name}
        row.update(metric_summary(y_true, y_pred))
        rows.append(row)
    df = pd.DataFrame(rows).set_index('model')
    return df[METRIC_ORDER].sort_values('macro_f1', ascending=False)


def mcnemar_test(y_true: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray, name_a: str, name_b: str) -> dict[str, float | str]:
    correct_a = pred_a == y_true
    correct_b = pred_b == y_true
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))

    if b + c == 0:
        chi2_stat = 0.0
        p_value = 1.0
    else:
        from scipy.stats import chi2
        chi2_stat = (abs(b - c) - 1) ** 2 / (b + c)
        p_value = 1.0 - chi2.cdf(chi2_stat, df=1)

    return {
        'comparison': f'{name_a} vs {name_b}',
        'a_correct_b_wrong': b,
        'a_wrong_b_correct': c,
        'chi2': chi2_stat,
        'p_value': p_value,
    }


def per_class_metrics_table(predictions: dict[str, np.ndarray], y_true: np.ndarray) -> pd.DataFrame:
    rows = []
    for model_name, y_pred in predictions.items():
        report = classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            output_dict=True,
            zero_division=0,
        )
        for label in ['0', '1']:
            rows.append(
                {
                    'model': model_name,
                    'class': int(label),
                    'precision': report[label]['precision'],
                    'recall': report[label]['recall'],
                    'f1': report[label]['f1-score'],
                    'support': int(report[label]['support']),
                }
            )
    return pd.DataFrame(rows).set_index(['model', 'class'])


def confusion_matrix_table(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return pd.DataFrame(cm, index=['gold_0', 'gold_1'], columns=['pred_0', 'pred_1'])


official_baseline_df = pd.read_csv(OFFICIAL_BASELINE_PATH)
official_baseline_df = official_baseline_df.rename(columns=lambda col: str(col).strip())
if 'Unnamed: 0' in official_baseline_df.columns:
    official_baseline_df = official_baseline_df.drop(columns=['Unnamed: 0'])

official_baseline_df['reference'] = official_baseline_df['reference'].astype(int)
if official_baseline_df['reference'].tolist() != dev_df['label'].tolist():
    raise ValueError('The reference column in 25_DEV_NLI.csv does not match training_data/NLI/dev.csv.')

official_svm_pred = official_baseline_df['SVM'].astype(int).to_numpy()
y_dev = dev_df['label'].to_numpy(dtype=int)
y_train = train_df['label'].to_numpy(dtype=int)

print('Official baseline methods available:', [col for col in official_baseline_df.columns if col != 'reference'])
pd.DataFrame({'reference': official_baseline_df['reference'].head(5), 'SVM': official_baseline_df['SVM'].head(5)})


Official baseline methods available: ['SVM', 'LSTM', 'BERT']


,reference,SVM
0,0,0
1,1,1
2,1,1
3,0,0
4,1,1


## 5. Final Solution A representation

The main Solution A representation stays inside traditional machine learning, but is richer than the internal baseline because it adds pairwise structure.

Feature blocks used by the rich representation:

1. **Premise word TF-IDF**: word 1-2 grams from the premise only
2. **Hypothesis word TF-IDF**: word 1-2 grams from the hypothesis only
3. **Shared-space interactions**: absolute difference and element-wise product after projecting both texts into the same word-TF-IDF space
4. **Pair-level character TF-IDF**: character n-grams from `premise [SEP] hypothesis`
5. **Hand-crafted pair features** computed directly from the provided text:
   - lexical overlap
   - new-token ratio
   - length features
   - negation mismatch
   - number mismatch
   - simple punctuation cues

The richer representation supports several Category A candidates: Logistic Regression, Linear SVM, ablations, bootstrap LR ensembles, and vocabulary-size sensitivity checks.


In [5]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")
NUMBER_PATTERN = re.compile(r'\d+(?:\.\d+)?')
NEGATION_TOKENS = {'no', 'not', 'never', 'none', 'nobody', 'nothing', 'neither', 'nor', 'without'}
FULL_FEATURE_BLOCK_ORDER = [
    'premise_word_tfidf',
    'hypothesis_word_tfidf',
    'shared_abs_difference',
    'shared_product',
    'pair_char_tfidf',
    'handcrafted_dense',
]


def normalize_text(text: str) -> str:
    text = str(text)
    text = text.replace('’', "'").replace('‘', "'")
    text = text.replace('“', '"').replace('”', '"')
    return re.sub(r'\s+', ' ', text).strip()


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(normalize_text(text).lower())


HANDCRAFTED_FEATURE_NAMES = [
    'hypothesis_token_recall',
    'premise_token_precision',
    'jaccard',
    'new_token_ratio',
    'premise_length',
    'hypothesis_length',
    'length_ratio',
    'length_difference',
    'premise_has_negation',
    'hypothesis_has_negation',
    'negation_mismatch',
    'shared_number_count',
    'number_mismatch',
    'exact_string_match',
    'hypothesis_token_subset',
    'premise_has_question_mark',
    'hypothesis_has_question_mark',
    'question_mark_delta',
    'premise_has_exclamation_mark',
    'hypothesis_has_exclamation_mark',
    'exclamation_mark_delta',
]


def build_handcrafted_pair_features(df: pd.DataFrame) -> np.ndarray:
    rows = []
    for premise, hypothesis in zip(df['premise'], df['hypothesis']):
        premise_text = normalize_text(premise)
        hypothesis_text = normalize_text(hypothesis)
        premise_tokens = tokenize(premise_text)
        hypothesis_tokens = tokenize(hypothesis_text)
        premise_set = set(premise_tokens)
        hypothesis_set = set(hypothesis_tokens)

        overlap = len(premise_set & hypothesis_set)
        union = len(premise_set | hypothesis_set)
        hypothesis_token_recall = overlap / len(hypothesis_set) if hypothesis_set else 0.0
        premise_token_precision = overlap / len(premise_set) if premise_set else 0.0
        jaccard = overlap / union if union else 0.0
        new_token_ratio = len(hypothesis_set - premise_set) / len(hypothesis_set) if hypothesis_set else 0.0

        premise_length = len(premise_tokens)
        hypothesis_length = len(hypothesis_tokens)
        length_ratio = hypothesis_length / premise_length if premise_length else 0.0
        length_difference = premise_length - hypothesis_length

        premise_has_negation = int(any(tok in NEGATION_TOKENS or tok.endswith("n't") for tok in premise_tokens))
        hypothesis_has_negation = int(any(tok in NEGATION_TOKENS or tok.endswith("n't") for tok in hypothesis_tokens))
        negation_mismatch = int(premise_has_negation != hypothesis_has_negation)

        premise_numbers = NUMBER_PATTERN.findall(premise_text)
        hypothesis_numbers = NUMBER_PATTERN.findall(hypothesis_text)
        shared_number_count = len(set(premise_numbers) & set(hypothesis_numbers))
        number_mismatch = int(bool(premise_numbers or hypothesis_numbers) and set(premise_numbers) != set(hypothesis_numbers))

        exact_string_match = int(premise_text.lower() == hypothesis_text.lower())
        hypothesis_token_subset = int(hypothesis_set.issubset(premise_set)) if hypothesis_set else 0

        premise_has_question_mark = int('?' in premise_text)
        hypothesis_has_question_mark = int('?' in hypothesis_text)
        question_mark_delta = hypothesis_has_question_mark - premise_has_question_mark
        premise_has_exclamation_mark = int('!' in premise_text)
        hypothesis_has_exclamation_mark = int('!' in hypothesis_text)
        exclamation_mark_delta = hypothesis_has_exclamation_mark - premise_has_exclamation_mark

        rows.append([
            hypothesis_token_recall,
            premise_token_precision,
            jaccard,
            new_token_ratio,
            premise_length,
            hypothesis_length,
            length_ratio,
            length_difference,
            premise_has_negation,
            hypothesis_has_negation,
            negation_mismatch,
            shared_number_count,
            number_mismatch,
            exact_string_match,
            hypothesis_token_subset,
            premise_has_question_mark,
            hypothesis_has_question_mark,
            question_mark_delta,
            premise_has_exclamation_mark,
            hypothesis_has_exclamation_mark,
            exclamation_mark_delta,
        ])

    return np.asarray(rows, dtype=np.float32)


def sparse_absolute_difference(left: sp.csr_matrix, right: sp.csr_matrix) -> sp.csr_matrix:
    diff = (left - right).tocsr(copy=True)
    diff.data = np.abs(diff.data)
    return diff


In [6]:
class SolutionAFeatureBuilder:
    def __init__(
        self,
        premise_word_max_features: int = 12000,
        hypothesis_word_max_features: int = 12000,
        shared_word_max_features: int = 8000,
        pair_char_max_features: int = 8000,
    ) -> None:
        self.premise_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=premise_word_max_features,
            sublinear_tf=True,
        )
        self.hypothesis_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=hypothesis_word_max_features,
            sublinear_tf=True,
        )
        self.shared_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=shared_word_max_features,
            sublinear_tf=True,
        )
        self.pair_char_vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=2,
            max_features=pair_char_max_features,
            sublinear_tf=True,
        )
        self.handcrafted_scaler = StandardScaler()
        self.tfidf_scaler = MaxAbsScaler()
        self.handcrafted_feature_names_ = HANDCRAFTED_FEATURE_NAMES.copy()
        self.feature_block_dimensions_ = {}
        self.config_ = {
            'premise_word_max_features': premise_word_max_features,
            'hypothesis_word_max_features': hypothesis_word_max_features,
            'shared_word_max_features': shared_word_max_features,
            'pair_char_max_features': pair_char_max_features,
        }

    def _normalised_premise_series(self, df: pd.DataFrame) -> pd.Series:
        return df['premise'].fillna('').map(normalize_text)

    def _normalised_hypothesis_series(self, df: pd.DataFrame) -> pd.Series:
        return df['hypothesis'].fillna('').map(normalize_text)

    def fit(self, df: pd.DataFrame) -> 'SolutionAFeatureBuilder':
        premise_text = self._normalised_premise_series(df)
        hypothesis_text = self._normalised_hypothesis_series(df)
        pair_text = premise_text + ' [SEP] ' + hypothesis_text
        handcrafted = build_handcrafted_pair_features(df)

        self.premise_word_vectorizer.fit(premise_text)
        self.hypothesis_word_vectorizer.fit(hypothesis_text)
        self.shared_word_vectorizer.fit(pd.concat([premise_text, hypothesis_text], ignore_index=True))
        self.pair_char_vectorizer.fit(pair_text)
        self.handcrafted_scaler.fit(handcrafted)

        # Fit MaxAbsScaler on concatenated TF-IDF blocks so all sparse features
        # are on the same scale as the StandardScaler-normalised handcrafted features.
        _shared_prem = self.shared_word_vectorizer.transform(premise_text)
        _shared_hyp = self.shared_word_vectorizer.transform(hypothesis_text)
        _tfidf_combined = sp.hstack([
            self.premise_word_vectorizer.transform(premise_text),
            self.hypothesis_word_vectorizer.transform(hypothesis_text),
            sparse_absolute_difference(_shared_prem, _shared_hyp),
            _shared_prem.multiply(_shared_hyp),
            self.pair_char_vectorizer.transform(pair_text),
        ], format='csr')
        self.tfidf_scaler.fit(_tfidf_combined)

        self.feature_block_dimensions_ = {
            'premise_word_tfidf': len(self.premise_word_vectorizer.get_feature_names_out()),
            'hypothesis_word_tfidf': len(self.hypothesis_word_vectorizer.get_feature_names_out()),
            'shared_abs_difference': len(self.shared_word_vectorizer.get_feature_names_out()),
            'shared_product': len(self.shared_word_vectorizer.get_feature_names_out()),
            'pair_char_tfidf': len(self.pair_char_vectorizer.get_feature_names_out()),
            'handcrafted_dense': len(self.handcrafted_feature_names_),
        }
        return self

    def transform(self, df: pd.DataFrame) -> sp.csr_matrix:
        blocks = build_feature_block_matrices(feature_components_from_builder(self), df)
        return stack_selected_blocks(blocks, FULL_FEATURE_BLOCK_ORDER)

    def fit_transform(self, df: pd.DataFrame) -> sp.csr_matrix:
        self.fit(df)
        return self.transform(df)


def feature_components_from_builder(feature_builder: SolutionAFeatureBuilder) -> dict:
    return {
        'premise_word_vectorizer': feature_builder.premise_word_vectorizer,
        'hypothesis_word_vectorizer': feature_builder.hypothesis_word_vectorizer,
        'shared_word_vectorizer': feature_builder.shared_word_vectorizer,
        'pair_char_vectorizer': feature_builder.pair_char_vectorizer,
        'handcrafted_scaler': feature_builder.handcrafted_scaler,
        'tfidf_scaler': feature_builder.tfidf_scaler,
        'handcrafted_feature_names': feature_builder.handcrafted_feature_names_,
        'feature_block_dimensions': feature_builder.feature_block_dimensions_,
        'feature_config': feature_builder.config_,
    }


def build_feature_block_matrices(feature_components: dict, df: pd.DataFrame) -> dict[str, sp.csr_matrix]:
    premise_text = df['premise'].fillna('').map(normalize_text)
    hypothesis_text = df['hypothesis'].fillna('').map(normalize_text)
    pair_text = premise_text + ' [SEP] ' + hypothesis_text

    premise_word = feature_components['premise_word_vectorizer'].transform(premise_text)
    hypothesis_word = feature_components['hypothesis_word_vectorizer'].transform(hypothesis_text)

    shared_premise = feature_components['shared_word_vectorizer'].transform(premise_text)
    shared_hypothesis = feature_components['shared_word_vectorizer'].transform(hypothesis_text)
    shared_abs_difference = sparse_absolute_difference(shared_premise, shared_hypothesis)
    shared_product = shared_premise.multiply(shared_hypothesis)

    pair_char = feature_components['pair_char_vectorizer'].transform(pair_text)
    handcrafted = build_handcrafted_pair_features(df)
    handcrafted_scaled = feature_components['handcrafted_scaler'].transform(handcrafted)

    if 'tfidf_scaler' in feature_components:
        _tfidf_combined = sp.hstack(
            [premise_word, hypothesis_word, shared_abs_difference, shared_product, pair_char],
            format='csr',
        )
        _tfidf_scaled = feature_components['tfidf_scaler'].transform(_tfidf_combined)
        _splits = [
            premise_word.shape[1],
            premise_word.shape[1] + hypothesis_word.shape[1],
            premise_word.shape[1] + hypothesis_word.shape[1] + shared_abs_difference.shape[1],
            premise_word.shape[1] + hypothesis_word.shape[1] + 2 * shared_abs_difference.shape[1],
        ]
        premise_word = _tfidf_scaled[:, :_splits[0]]
        hypothesis_word = _tfidf_scaled[:, _splits[0]:_splits[1]]
        shared_abs_difference = _tfidf_scaled[:, _splits[1]:_splits[2]]
        shared_product = _tfidf_scaled[:, _splits[2]:_splits[3]]
        pair_char = _tfidf_scaled[:, _splits[3]:]

    return {
        'premise_word_tfidf': premise_word,
        'hypothesis_word_tfidf': hypothesis_word,
        'shared_abs_difference': shared_abs_difference,
        'shared_product': shared_product,
        'pair_char_tfidf': pair_char,
        'handcrafted_dense': sp.csr_matrix(handcrafted_scaled),
    }


def stack_selected_blocks(blocks: dict[str, sp.csr_matrix], selected_blocks: list[str]) -> sp.csr_matrix:
    return sp.hstack([blocks[block_name] for block_name in selected_blocks], format='csr')


def transform_with_feature_components(
    feature_components: dict,
    df: pd.DataFrame,
    selected_blocks: list[str] | None = None,
) -> sp.csr_matrix:
    blocks = build_feature_block_matrices(feature_components, df)
    if selected_blocks is None:
        selected_blocks = FULL_FEATURE_BLOCK_ORDER
    return stack_selected_blocks(blocks, selected_blocks)


In [7]:
SOLUTION_A_BUNDLE_PATH = ARTEFACT_DIR / 'nli_solution_a_bundle.joblib'
SOLUTION_A_METRICS_PATH = ARTEFACT_DIR / 'nli_solution_a_dev_metrics.csv'

In [8]:
def load_solution_a_bundle(bundle_path: Path = SOLUTION_A_BUNDLE_PATH) -> dict:
    return joblib.load(bundle_path)


def predict_from_solution_a_bundle(bundle: dict, input_df: pd.DataFrame) -> np.ndarray:
    X_input = transform_with_feature_components(
        bundle['feature_components'],
        input_df,
        selected_blocks=bundle['selected_blocks'],
    )

    if bundle['predictor_type'] == 'single_estimator':
        return bundle['estimator'].predict(X_input).astype(int)

    if bundle['predictor_type'] == 'bootstrap_lr_ensemble':
        probability_sum = None
        for estimator in bundle['estimators']:
            model_proba = estimator.predict_proba(X_input)
            probability_sum = model_proba if probability_sum is None else probability_sum + model_proba
        average_proba = probability_sum / len(bundle['estimators'])
        return bundle['classes'][np.argmax(average_proba, axis=1)].astype(int)

    raise ValueError(f"Unsupported predictor type: {bundle['predictor_type']}")


def predict_with_solution_a(
    input_csv: Path,
    output_csv: Path,
    bundle_path: Path = SOLUTION_A_BUNDLE_PATH,
) -> tuple[pd.DataFrame, np.ndarray, Path]:
    bundle = load_solution_a_bundle(bundle_path)
    input_df = read_pair_dataframe(Path(input_csv), require_label=False)
    predictions = predict_from_solution_a_bundle(bundle, input_df)

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.Series(predictions, name='label').to_csv(output_csv, index=False, header=False)

    return input_df, predictions, output_csv


## Load saved model and evaluate on dev set

Run `solution_A_train.ipynb` first to generate the bundle and metrics CSV.

In [9]:
bundle = load_solution_a_bundle(SOLUTION_A_BUNDLE_PATH)
best_deployable_candidate_name = bundle['metadata']['selected_final_candidate']
best_dev_pred = predict_from_solution_a_bundle(bundle, dev_df)

print('Selected model:', best_deployable_candidate_name)
print()
pd.DataFrame([metric_summary(y_dev, best_dev_pred)], index=[best_deployable_candidate_name]).T

Selected model: Rich-feature Logistic Regression



,Rich-feature Logistic Regression
accuracy_score,0.688836
macro_precision,0.688748
macro_recall,0.687718
macro_f1,0.687832
weighted_macro_precision,0.688784
weighted_macro_recall,0.688836
weighted_mmacro_f1,0.688410
matthews_corrcoef,0.376464


In [10]:
if SOLUTION_A_METRICS_PATH.exists():
    comparison_df = pd.read_csv(SOLUTION_A_METRICS_PATH, index_col=0)
    print('All candidate models — dev set metrics (sorted by macro F1):')
    comparison_df.sort_values('macro_f1', ascending=False)
else:
    print('Metrics CSV not found. Run solution_A_train.ipynb first.')

All candidate models — dev set metrics (sorted by macro F1):


## 8. Confusion matrices and per-class metrics

These outputs make the evaluation section more informative than a single summary score.

- **Confusion matrices** show where each model makes different kinds of mistakes.
- **Per-class precision / recall / F1** show whether a model is only strong on one label or is balanced across both.


In [11]:
selected_for_diagnostics = {
    best_deployable_candidate_name: best_dev_pred,
    'Official Category A baseline (SVM)': official_svm_pred,
}

confusion_tables = pd.concat(
    {name: confusion_matrix_table(y_dev, pred) for name, pred in selected_for_diagnostics.items()},
    names=['model', 'gold_label'],
)
per_class_df = per_class_metrics_table(selected_for_diagnostics, y_dev)

print('Confusion matrices:')
confusion_tables

Confusion matrices:


pred_0  pred_1
model                              gold_label                
Rich-feature Logistic Regression   gold_0        2129    1129
                                   gold_1         967    2511
Official Category A baseline (SVM) gold_0        1761    1497
                                   gold_1        1290    2188

In [12]:
print('Per-class precision / recall / F1:')
per_class_df


Per-class precision / recall / F1:


precision    recall        f1  \
model                              class                                  
Rich-feature Logistic Regression   0       0.687661  0.653468  0.670129   
                                   1       0.689835  0.721967  0.705535   
Official Category A baseline (SVM) 0       0.577188  0.540516  0.558250   
                                   1       0.593758  0.629097  0.610917   

                                          support  
model                              class           
Rich-feature Logistic Regression   0         3258  
                                   1         3478  
Official Category A baseline (SVM) 0         3258  
                                   1         3478

## 9. Structured error analysis

The goal here is not to produce a fully manual linguistic analysis.
Instead, this section creates reproducible slices that are useful later for a poster, model card, or oral explanation.

We look at:

- examples where the best deployable A model beats the official SVM baseline
- examples where all notebook A candidates still fail
- a brief pattern summary for negation mismatch, lexical-overlap traps, number mismatch, and long-sentence cases


In [13]:
analysis_df = dev_df.copy()
analysis_df[f'{best_deployable_candidate_name}__pred'] = best_dev_pred
analysis_df[f'{best_deployable_candidate_name}__correct'] = best_dev_pred == y_dev
analysis_df['Official Category A baseline (SVM)__pred'] = official_svm_pred
analysis_df['Official Category A baseline (SVM)__correct'] = official_svm_pred == y_dev

analysis_df['best_model_correct'] = analysis_df[f'{best_deployable_candidate_name}__correct']
analysis_df['official_svm_correct'] = analysis_df['Official Category A baseline (SVM)__correct']
analysis_df['best_model_fails'] = ~analysis_df['best_model_correct']

handcrafted_dev = pd.DataFrame(
    build_handcrafted_pair_features(dev_df),
    columns=HANDCRAFTED_FEATURE_NAMES,
    index=dev_df.index,
)
analysis_df = pd.concat([analysis_df, handcrafted_dev], axis=1)
analysis_df['lexical_overlap_trap'] = (analysis_df['jaccard'] >= 0.5) & (analysis_df['label'] == 0)
analysis_df['long_sentence_case'] = analysis_df[['premise_length', 'hypothesis_length']].max(axis=1) >= 18

best_beats_baseline_examples = analysis_df[
    analysis_df['best_model_correct'] & ~analysis_df['official_svm_correct']
][[
    'premise', 'hypothesis', 'label',
    'Official Category A baseline (SVM)__pred',
    f'{best_deployable_candidate_name}__pred',
]].head(5)

all_fail_examples = analysis_df[analysis_df['best_model_fails']][[
    'premise', 'hypothesis', 'label',
    'Official Category A baseline (SVM)__pred',
    f'{best_deployable_candidate_name}__pred',
]].head(5)

pattern_specs = {
    'negation_mismatch_cases': analysis_df['negation_mismatch'] == 1,
    'lexical_overlap_traps': analysis_df['lexical_overlap_trap'],
    'number_mismatch_cases': analysis_df['number_mismatch'] == 1,
    'long_sentence_cases': analysis_df['long_sentence_case'],
}

pattern_rows = []
for pattern_name, mask in pattern_specs.items():
    subset = analysis_df[mask]
    if subset.empty:
        pattern_rows.append({'pattern': pattern_name, 'count': 0,
                             'best_model_accuracy': float('nan'), 'official_svm_accuracy': float('nan')})
        continue
    pattern_rows.append({
        'pattern': pattern_name,
        'count': int(mask.sum()),
        'best_model_accuracy': subset['best_model_correct'].mean(),
        'official_svm_accuracy': subset['official_svm_correct'].mean(),
    })

pattern_summary_df = pd.DataFrame(pattern_rows).set_index('pattern')

print('Examples where the best model beats the official SVM baseline:')
best_beats_baseline_examples

Examples where the best model beats the official SVM baseline:


,premise,hypothesis,label,Official Category A baseline (SVM)__pred,Rich-feature Logistic Regression__pred
10,"The festival sounds woke me up, and the smell ...",The festival was vegetarian.,0,1,0
11,"Corporate social responsibility CSR, also call...","Corporate social responsibility (CSR, also cal...",1,0,1
16,"Once or twice, but they seem more show than ba...",Adrin said they were amazing warriors.,0,1,0
21,Or anything else you wanted and couldn't keep ...,Magic had little power.,0,1,0
24,"""With cattle horses teaming?""","""Teaming with cattle horses near the desert?""",1,0,1


In [14]:
print('Examples where all notebook A candidates still fail:')
all_fail_examples


Examples where all notebook A candidates still fail:


,premise,hypothesis,label,Official Category A baseline (SVM)__pred,Rich-feature Logistic Regression__pred
3,A man with a black shirt holds a baby while a ...,A darkly dressed man passes a crying baby to a...,0,0,1
7,um-hum oh i i guess i didn't hear that i didn'...,I didn't hear the speech.,1,0,0
9,"Putting ethics aside, journalists tend to exte...",Journalists attempt to make the conflicts shor...,0,0,1
13,Jane Finn.,A young girl.,1,1,0
14,I would never have dreamt of suspecting the do...,"The doctor didn't do it, and everybody was cer...",0,0,1


In [15]:
print('Pattern summary for later write-up use:')
pattern_summary_df


Pattern summary for later write-up use:


,count,best_model_accuracy,official_svm_accuracy
pattern,,,
negation_mismatch_cases,1935,0.739535,0.631525
lexical_overlap_traps,260,0.742308,0.446154
number_mismatch_cases,872,0.667431,0.580275
long_sentence_cases,3299,0.689906,0.579570
